In [1]:
# Imports
import sys
import logging
from datetime import datetime
import pandas as pd
from IPython.display import display

sys.path.insert(0, '../../../LOGOS')
from src import Pert, plot_gantt_chart, plot_resource_utilization, plot_location_utilization, plot_equipment_utilization
# Configure logging in the runner (avoid setting basicConfig inside the module)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
import json
from pathlib import Path

cwd = Path.cwd()
benchmark = cwd/'benchmarks'
benchmark_results_file = benchmark/'priority_rules_results.json'

## Load Benchmark results
with open(benchmark_results_file, "r", encoding="utf-8") as f:
    benchmark_data = json.load(f)

In [2]:
def run_case(case_name, file_name, json_path,
             schema_file="outage_schema.json",
             benchmark_data=None):
    """
    Runs scheduling comparison for a given case and file.

    Parameters:
        case_name (str): Case identifier (e.g., 'j60')
        file_name (str): File name (e.g., 'j601_1.sm')
        json_path (str): Path to JSON file for Pert model
        schema_file (str): Path to schema file (default: outage_schema.json)
        benchmark_data (dict): Benchmark dataset for RCPSP comparison

    Returns:
        results_df (pd.DataFrame): LOGOS.CPM results
        data_df (pd.DataFrame): RCPSP benchmark results
    """

    results = {}

    # Load Pert Model
    pert = Pert.from_json_file(json_path, schema_path=schema_file)

    prs = [
        'lf', 'ls', 'ef', 'es', 'duration', 'random',
        'mts', 'mtp', 'grpw', 'grd', 'rr', 'avgrr',
        'maxrr', 'minrr','mehh_8000_b','mehh_3375_b',
        'mehh_1000_b','mehh_125_b','gphh_b'
    ]

    # Compute results
    for rule in prs:
        out = pert.calculateSerialScheduleWithResources(priority_rule=rule)
        results[rule] = out['scheduled_duration'] - 2  # remove start/end duration

    print('Results from LOGOS.CPM:')
    print('-' * 60)
    results_df = pd.DataFrame(results, index=[0])
    display(results_df)

    # RCPSP benchmark comparison
    print('Results from RCPSP')
    print('-' * 60)

    if benchmark_data is None:
        raise ValueError("benchmark_data must be provided")

    data = benchmark_data[case_name][file_name]
    data_df = pd.DataFrame(data, index=[0]).filter(like='serial_forward')
    data_df.columns = data_df.columns.str.replace("_serial_forward", "", regex=False)
    data_df.columns = data_df.columns.str.lower()

    display(data_df)

    return results_df, data_df

## Scheduling with 30 activities

In [3]:
results_df, data_df = run_case(
    case_name='j30',
    file_name='j301_1.sm',
    json_path='j301_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=32 | CPM=40.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 07:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J8 | start=2026-01-01 05:00 | end=2026-01-01 14:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J10 | start=2026-01-01 07:00 | end=2026-01-01 14:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 05:00 | end=2026-01-01 13:00 | delay=4.0h
DEBUG:root:Serial SGS: scheduled J9 | start=2026-01-01 07:00 | end=2026-01-01 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J12 | start=2026-01-01 14:00 | end=2026-01-01 16:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-01 09:00 | end=2026-01-01 15:

DEBUG:root:Serial SGS: scheduled J18 | start=2026-01-01 15:00 | end=2026-01-01 20:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J7 | start=2026-01-01 12:00 | end=2026-01-01 17:00 | delay=7.0h
DEBUG:root:Serial SGS: scheduled J19 | start=2026-01-01 19:00 | end=2026-01-01 22:00 | delay=5.0h
DEBUG:root:Serial SGS: scheduled J20 | start=2026-01-01 22:00 | end=2026-01-02 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J22 | start=2026-01-02 06:00 | end=2026-01-02 13:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J15 | start=2026-01-01 13:00 | end=2026-01-01 22:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J21 | start=2026-01-02 06:00 | end=2026-01-02 08:00 | delay=6.0h
DEBUG:root:Serial SGS: scheduled J23 | start=2026-01-02 13:00 | end=2026-01-02 15:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J27 | start=2026-01-02 08:00 | end=2026-01-02 16:00 | delay=15.0h
DEBUG:root:Serial SGS: scheduled J6 | start=2026-01-02 16:00 | end=2026-01-03 00:00 | delay=27.0h
DEBUG:root:Serial SG


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED


INFO:root:Starting Serial SGS | activities=32 | CPM=40.0h | rule=mehh_125_b
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 07:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J16 | start=2026-01-01 14:00 | end=2026-01-02 00:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 05:00 | end=2026-01-01 13:00 | delay=4.0h
DEBUG:root:Serial SGS: scheduled J11 | start=2026-01-01 13:00 | end=2026-01-01 22:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J15 | start=2026-01-01 13:00 | end=2026-01-01 22:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J8 | start=2026-01-01 05:00 | end=2026-01-01 14:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J6 | start=2026-01-02 00:00 | end=2026-01-02 08:00 | delay=11.0h
DEBUG:root:Serial SGS: scheduled J27 | start=20

Results from LOGOS.CPM:
------------------------------------------------------------


,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,49.0,46.0,60.0,51.0,44.0,49.0,49.0,44.0,44.0,45.0,49.0,45.0,45.0,49.0,52.0,50.0,52.0,43.0,44.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,51,60,46,49,57,49,49,61,60,53,52,53,52,46,74



## Scheduling with 60 activities

In [4]:
results_df, data_df = run_case(
    case_name='j60',
    file_name='j601_1.sm',
    json_path='j601_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=62 | CPM=79.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 11:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J8 | start=2026-01-01 11:00 | end=2026-01-01 20:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J9 | start=2026-01-01 20:00 | end=2026-01-01 21:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 02:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-01 21:00 | end=2026-01-02 03:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J14 | start=2026-01-01 02:00 | end=2026-01-01 04:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 01:00 | end=2026-01-01 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J18 | start=2026-01-02 03:00 | end=2026-01-02 13:


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED


DEBUG:root:Serial SGS: scheduled J45 | start=2026-01-01 20:00 | end=2026-01-02 02:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-02 21:00 | end=2026-01-03 03:00 | delay=24.0h
DEBUG:root:Serial SGS: scheduled J40 | start=2026-01-02 02:00 | end=2026-01-02 11:00 | delay=5.0h
DEBUG:root:Serial SGS: scheduled J15 | start=2026-01-01 09:00 | end=2026-01-01 14:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J33 | start=2026-01-02 13:00 | end=2026-01-02 19:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J46 | start=2026-01-02 02:00 | end=2026-01-02 08:00 | delay=6.0h
DEBUG:root:Serial SGS: scheduled J60 | start=2026-01-02 11:00 | end=2026-01-02 21:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J28 | start=2026-01-02 13:00 | end=2026-01-02 22:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J47 | start=2026-01-02 22:00 | end=2026-01-03 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J11 | start=2026-01-02 08:00 | end=2026-01-02 16:00 | delay=12.0h
DEBUG:root:Serial 

Results from LOGOS.CPM:
------------------------------------------------------------


,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,77.0,77.0,88.0,86.0,77.0,77.0,77.0,77.0,77.0,77.0,80.0,77.0,77.0,80.0,77.0,77.0,77.0,77.0,83.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,86,88,77,77,121,80,77,106,84,98,77,85,77,109,121


## Scheduling with 90 activities

In [5]:
results_df, data_df = run_case(
    case_name='j90',
    file_name='j901_1.sm',
    json_path='j901_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=92 | CPM=69.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 11:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 01:00 | end=2026-01-01 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 02:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J15 | start=2026-01-01 11:00 | end=2026-01-01 18:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J20 | start=2026-01-01 18:00 | end=2026-01-01 21:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J11 | start=2026-01-01 09:00 | end=2026-01-01 17:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J12 | start=2026-01-01 17:00 | end=2026-01-01 19:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J5 | start=2026-01-01 02:00 | end=2026-01-01 04


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED


DEBUG:root:Serial SGS: scheduled J19 | start=2026-01-01 11:00 | end=2026-01-01 20:00 | delay=9.0h
DEBUG:root:Serial SGS: scheduled J39 | start=2026-01-02 14:00 | end=2026-01-02 19:00 | delay=23.0h
DEBUG:root:Serial SGS: scheduled J43 | start=2026-01-02 04:00 | end=2026-01-02 12:00 | delay=2.0h
DEBUG:root:Serial SGS: scheduled J45 | start=2026-01-01 19:00 | end=2026-01-01 20:00 | delay=2.0h
DEBUG:root:Serial SGS: scheduled J49 | start=2026-01-02 14:00 | end=2026-01-02 15:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J51 | start=2026-01-02 15:00 | end=2026-01-02 23:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J61 | start=2026-01-02 22:00 | end=2026-01-03 01:00 | delay=6.0h
DEBUG:root:Serial SGS: scheduled J64 | start=2026-01-02 23:00 | end=2026-01-03 04:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J70 | start=2026-01-02 23:00 | end=2026-01-03 06:00 | delay=1.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-02 18:00 | end=2026-01-03 01:00 | delay=37.0h
DEBUG:root:Serial 

Results from LOGOS.CPM:
------------------------------------------------------------


,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,82.0,83.0,98.0,94.0,87.0,84.0,89.0,80.0,81.0,77.0,88.0,99.0,99.0,88.0,83.0,90.0,81.0,80.0,80.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,94,98,83,82,111,88,89,101,95,108,84,101,84,102,148


## Scheduling with 120 activities

In [6]:
results_df, data_df = run_case(
    case_name='j120',
    file_name='j1201_1.sm',
    json_path='j1201_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=122 | CPM=101.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J6 | start=2026-01-01 05:00 | end=2026-01-01 08:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J7 | start=2026-01-01 08:00 | end=2026-01-01 18:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J11 | start=2026-01-01 18:00 | end=2026-01-02 00:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 03:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-01 03:00 | end=2026-01-01 10:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J18 | start=2026-01-02 00:00 | end=2026-01-02 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J8 | start=2026-01-01 03:00 | end=2026-01-01 0


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED


DEBUG:root:Serial SGS: scheduled J98 | start=2026-01-03 22:00 | end=2026-01-04 03:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J99 | start=2026-01-02 02:00 | end=2026-01-02 07:00 | delay=2.0h
DEBUG:root:Serial SGS: scheduled J102 | start=2026-01-04 08:00 | end=2026-01-04 13:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J114 | start=2026-01-04 03:00 | end=2026-01-04 08:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 19:00 | end=2026-01-02 01:00 | delay=18.0h
DEBUG:root:Serial SGS: scheduled J11 | start=2026-01-01 20:00 | end=2026-01-02 02:00 | delay=2.0h
DEBUG:root:Serial SGS: scheduled J32 | start=2026-01-01 15:00 | end=2026-01-01 21:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J35 | start=2026-01-02 07:00 | end=2026-01-02 13:00 | delay=23.0h
DEBUG:root:Serial SGS: scheduled J70 | start=2026-01-02 09:00 | end=2026-01-02 15:00 | delay=14.0h
DEBUG:root:Serial SGS: scheduled J73 | start=2026-01-02 09:00 | end=2026-01-02 15:00 | delay=0.0h
DEBUG:root:Seria

Results from LOGOS.CPM:
------------------------------------------------------------


,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,123.0,119.0,144.0,132.0,124.0,125.0,125.0,133.0,128.0,107.0,123.0,117.0,117.0,123.0,124.0,125.0,124.0,110.0,126.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,132,144,119,123,147,123,125,153,140,156,124,138,114,157,193
